# Movie Recommendation System using Collaborative Filtering

## Data Science Internship Project

This project builds a movie recommendation system using collaborative filtering. The system recommends movies to users based on similar user ratings.


In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## Load the Dataset



In [3]:
df = pd.read_csv('ratings.csv')
df.head()

,user_id,movie_id,rating,timestamp
0,7,25,3.4,1618227179
1,20,10,3.2,1643959985
2,15,22,1.9,1686261879
3,11,26,4.8,1503609736
4,8,3,4.1,1575169697


## Explore the Dataset

In [5]:
print('Dataset Shape:', df.shape)
print('\nDataset Info:')
print(df.info())

print('\nSummary Statistics:')
print(df.describe())

Dataset Shape: (300, 4)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   user_id    300 non-null    int64  
 1   movie_id   300 non-null    int64  
 2   rating     300 non-null    float64
 3   timestamp  300 non-null    int64  
dtypes: float64(1), int64(3)
memory usage: 9.5 KB
None

Summary Statistics:
          user_id    movie_id      rating     timestamp
count  300.000000  300.000000  300.000000  3.000000e+02
mean     9.930000   16.250000    3.020000  1.603292e+09
std      6.088404    8.890687    1.142461  5.588312e+07
min      1.000000    1.000000    1.000000  1.502156e+09
25%      4.000000    9.000000    2.000000  1.561895e+09
50%      9.000000   17.000000    3.100000  1.605533e+09
75%     16.000000   24.000000    4.000000  1.648004e+09
max     20.000000   30.000000    5.000000  1.699392e+09


## Create User-Item Matrix

Rows = Users  
Columns = Movies  
Values = Ratings

In [7]:
user_movie_matrix = df.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
).fillna(0)

user_movie_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,21,22,23,24,25,26,27,28,29,30
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,3.4,0.0,0.0,0.0,1.1,4.3,1.5,0.0,...,3.25,0.0,0.000,0.0,3.9,4.100000,0.0,0.0,4.1,0.0
2,0.0,3.6,3.0,2.7,0.0,0.0,0.0,0.0,0.0,2.6,...,0.00,0.0,3.900,4.5,0.0,1.400000,0.0,0.0,0.0,4.6
3,3.8,0.0,0.0,0.0,2.0,0.0,1.8,0.0,0.0,4.7,...,0.00,0.0,2.825,0.0,3.0,2.266667,0.0,0.0,4.4,2.1
4,0.0,3.1,0.0,2.9,0.0,0.0,2.1,2.0,0.0,0.0,...,0.00,2.0,2.600,0.0,0.0,0.000000,0.0,2.5,1.6,0.0
5,0.0,0.0,0.0,0.0,3.9,0.0,0.0,0.0,0.0,4.4,...,0.00,3.3,2.250,2.6,0.0,2.850000,4.1,0.0,0.0,0.0


## Compute User Similarity Matrix

We use cosine similarity to measure similarity between users.

In [9]:
user_similarity = cosine_similarity(user_movie_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
user_id,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.118580,0.526894,0.303807,0.216587,0.457425,0.524465,0.498256,0.481173,0.517185,0.383421,0.460143,0.205433,0.382096,0.236626,0.241590,0.220558,0.509065,0.390584,0.279916
2,0.118580,1.000000,0.363269,0.393144,0.347399,0.487277,0.351839,0.491874,0.315734,0.235087,0.337312,0.242228,0.184740,0.435923,0.204101,0.528637,0.654430,0.250960,0.378386,0.381493
3,0.526894,0.363269,1.000000,0.392082,0.439169,0.582053,0.410398,0.559164,0.396224,0.325033,0.185362,0.556782,0.388745,0.252905,0.250182,0.321648,0.530863,0.312659,0.592670,0.570698
4,0.303807,0.393144,0.392082,1.000000,0.170874,0.239711,0.269991,0.288536,0.226988,0.452264,0.181642,0.421845,0.424186,0.055490,0.058030,0.489208,0.531220,0.225387,0.226878,0.351979
5,0.216587,0.347399,0.439169,0.170874,1.000000,0.449309,0.295600,0.556692,0.166420,0.300558,0.477802,0.401862,0.170162,0.344792,0.207037,0.169185,0.435939,0.411957,0.219581,0.377219


## Recommendation Function

In [20]:
def recommend_movies(user_id, top_n=5):

    # get similarity scores for the user
    similarity_scores = user_similarity_df.loc[user_id]

    # multiply similarity with user-movie matrix
    weighted_scores = np.dot(similarity_scores, user_movie_matrix)

    # convert to pandas series
    recommendations = pd.Series(
        weighted_scores,
        index=user_movie_matrix.columns
    )

    # remove movies already watched
    watched_movies = user_movie_matrix.loc[user_id]
    recommendations = recommendations[watched_movies == 0]

    return recommendations.sort_values(ascending=False).head(top_n)

## Get Movie Recommendations

In [23]:
recommend_movies(user_id=1)

movie_id
30    11.898363
22    10.830308
28    10.540987
24    10.249515
5     10.233077
dtype: float64

## Improved Recommendation (Unwatched Movies Only)

In [30]:
def recommend_movies_filtered(user_id, top_n=5):

    # movies already watched
    watched_movies = user_movie_matrix.loc[user_id]

    # movies not watched
    unwatched_movies = watched_movies[watched_movies == 0].index

    # similarity scores
    similarity_scores = user_similarity_df.loc[user_id]

    # weighted movie scores
    weighted_scores = np.dot(similarity_scores, user_movie_matrix)

    # convert to pandas series
    recommendations = pd.Series(
        weighted_scores,
        index=user_movie_matrix.columns
    )

    # filter only unwatched movies
    recommendations = recommendations[unwatched_movies]

    return recommendations.sort_values(ascending=False).head(top_n)

In [32]:
recommend_movies_filtered(1)

movie_id
30    11.898363
22    10.830308
28    10.540987
24    10.249515
5     10.233077
dtype: float64